# Amazon Bedrock AgentCore Runtime e AgentCore Memory Agent com Isolamento de Identidade usando Cognito Federated Identities

## Visão Geral

Este tutorial demonstra como implementar isolamento seguro de memória em agentes conversacionais usando o Amazon Bedrock AgentCore Memory com Cognito federated identities. Você construirá um agente habilitado com memória que particiona e isola automaticamente o histórico de conversas com base em credenciais de usuário autenticadas do Amazon Cognito Identity Pool, garantindo privacidade de dados e segurança em ambientes multi-tenant.

O isolamento de memória é um requisito de segurança crítico para agentes conversacionais em produção. Sem o isolamento adequado, os usuários poderiam potencialmente acessar o histórico de conversas de outros usuários, levando a violações de privacidade e vazamento de dados. 

A implementação demonstra como o AgentCore Memory se integra com o Amazon Cognito Identity Pools para aplicar automaticamente limites de memória com base em credenciais de usuário federadas, eliminando a necessidade de lógica de autorização personalizada no código da sua aplicação.

### Detalhes do Tutorial

| Informação          | Detalhes                                                         |
|---------------------|------------------------------------------------------------------|
| Tipo do tutorial    | Segurança & Gerenciamento de Identidade                          |
| Tipo de agente      | Agente Conversacional Único                                      |
| Framework de agente | Strands Agents                                                   |
| Modelo LLM          | Anthropic Claude 4.5 Haiku                                      |
| Recursos principais | Memory Isolation, Federated Identity, User Context               |
| SDK utilizado       | boto3, bedrock-agentcore, bedrock-agentcore-starter-toolkit      |

### O Que Você Vai Aprender

Neste tutorial, você aprenderá:
1. Como configurar Cognito Identity Pools para federated identity no AgentCore Memory
2. Como propagar a identidade do usuário através de credenciais temporárias da AWS
3. Como implementar memory hooks com credenciais federadas para particionamento seguro de dados
4. Como implantar e testar agentes multi-usuário com espaços de memória isolados
5. Como verificar o isolamento de memória entre diferentes usuários usando federated identities


### Arquitetura

Este exemplo demonstra um agente conversacional implantado no AgentCore Runtime com isolamento de memória baseado em federated identity:

<div style="text-align:left">
    <img src="architecture.png" width="90%"/>
</div>


## 0. Pré-requisitos

Para executar este tutorial você precisará de:
* Python 3.10 ou mais recente
* Credenciais AWS configuradas com permissões apropriadas para Bedrock, ECR, IAM e Cognito
* Acesso ao modelo Amazon Bedrock (Claude 3.5 Haiku)
* Amazon Bedrock AgentCore SDK e dependências

Primeiro, vamos instalar as bibliotecas necessárias.

In [ ]:
!pip install -qUr requirements.txt

### Configurando o Ambiente

Vamos importar as bibliotecas necessárias e configurar nosso ambiente. Utilizaremos:
- `boto3` para interações com serviços AWS
- `bedrock_agentcore.memory` para gerenciar a memória do agente
- Diversas funções utilitárias para configurar a autenticação

In [ ]:
# Imports
import os
import boto3
import uuid
import logging
from bedrock_agentcore_starter_toolkit.operations.memory.manager import MemoryManager
from utils import setup_cognito_user_pool, create_agentcore_role

# Configuration
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")
REGION = os.getenv('AWS_REGION', 'us-west-2')

## 1. Criando o Recurso de Memória

Nesta seção, criaremos um recurso de memória para nosso agente armazenar o histórico de conversas. A memória permite que o agente relembre interações passadas, mantenha o contexto e forneça respostas mais coerentes ao longo do tempo.

Para este exemplo, criaremos um recurso de memória de curto prazo simples sem estratégias adicionais de longo prazo. A memória armazenará todas as mensagens de conversa, ajudando nosso agente a lembrar interações anteriores ao continuar uma sessão após ela ter sido encerrada no AgentCore Runtime.

In [ ]:
# Create unique identifier for this resource
unique_id = str(uuid.uuid4())[:8]
memory_name = f"RuntimeIdentityMemoryAgent_{unique_id}"

# Initialize Memory Manager
memory_manager = MemoryManager(region_name=REGION)

# Create memory
print("\n🧠 Creating memory...")
print("   This takes 2-3 minutes...\n")

memory = memory_manager.get_or_create_memory(
    name=memory_name,
    strategies=[],
    description="Memory isolation with IAM example.",
    event_expiry_days=30  # Optional: adjust as needed
)

MEMORY_ID = memory.get("id")
print(f"\n✅ Memory created successfully!")
print(f"   Memory ID: {MEMORY_ID}")
print(f"   Status: {memory.get('status')}")

## 2. Criando o Amazon Cognito User Pool e Identity Pool

Nesta seção, criaremos um Amazon Cognito User Pool, Identity Pool e usuários. O Cognito fornece autenticação de usuário e gerenciamento de identidade para nosso agente, garantindo que o histórico de conversas de cada usuário seja acessível apenas por aquele usuário através de credenciais de federated identity.

A função `setup_cognito_user_pool` irá:
1. Criar um Cognito User Pool se ele não existir
2. Criar um Cognito Identity Pool para federated identity
3. Configurar app clients para autenticação
4. Criar 2 usuários de teste com senhas temporárias
5. Gerar access tokens e ID tokens para testes

In [ ]:
print("Setting up Amazon Cognito user pool and users...")
cognito_config = setup_cognito_user_pool(REGION,MEMORY_ID)
print("Cognito setup completed ✓")

## 3. Criando Seu Agente Habilitado com Memória

Nesta seção, construiremos nosso agente habilitado com memória usando o framework Strands Agents com hooks personalizados para integração de memória. Este agente manterá o contexto da conversa armazenando e recuperando mensagens do AgentCore Memory usando credenciais de federated identity.

> **Por Que a Memória é Importante**: Sessões no AgentCore Runtime expiram após um certo tempo, o que apaga o contexto da conversa. Ao armazenar conversas na memória, garantimos que informações anteriores persistam entre sessões, criando uma experiência contínua para os usuários mesmo após longas pausas.

### Capacidades do Agente

Nosso agente irá:
1. Armazenar automaticamente cada mensagem do usuário e do assistente na memória
2. Recuperar o histórico de conversas anteriores ao continuar uma sessão existente
3. Manter o contexto ao longo de múltiplas interações com o mesmo usuário
4. Isolar conversas entre diferentes usuários através da verificação de federated identity

### Componentes Principais da Nossa Implementação

#### 1. Memory Hook Provider
Nosso hook provider personalizado implementa:
- `on_agent_initialized`: Disparado quando o agente inicia, recupera o histórico de conversas do AgentCore Memory
- `on_message_added`: Disparado quando uma nova mensagem é adicionada à conversa, armazena-a no AgentCore Memory

#### 2. Inicialização do Agente
A função `initialize_agent`:
- Configura o memory hook com a região correta
- Configura o agente com as variáveis de estado apropriadas (memory_id, actor_id, session_id)
- Configura o system prompt para o LLM

#### 3. Credenciais Federadas
A função `get_aws_credentials_for_identity`:
- Troca o Cognito ID token por credenciais temporárias da AWS
- Retorna credenciais que podem ser usadas para acessar serviços AWS com permissões específicas do usuário

#### 4. Handler do Entry Point
A função runtime_memory_agent:
- Analisa o payload de entrada e extrai a mensagem do usuário e o ID token
- Obtém credenciais federadas usando o ID token
- Gerencia a inicialização do agente e o rastreamento de sessão
- Trata a invocação do agente com o contexto apropriado
- Retorna respostas formatadas para o ambiente de runtime

Vamos criar o arquivo do nosso agente:

In [ ]:
%%writefile runtime_identity_memory_agent.py
import os
import jwt
import ast
import json
import boto3
import logging
import datetime

from strands import Agent
from jwt import PyJWKClient
from typing import Dict, Any
from strands.models import BedrockModel
from bedrock_agentcore.memory.session import MemorySessionManager
from bedrock_agentcore.runtime import BedrockAgentCoreApp
from strands.hooks import AgentInitializedEvent, HookProvider, HookRegistry, MessageAddedEvent
from bedrock_agentcore.memory.constants import StrategyType, ConversationalMessage, MessageRole

# Configure detailed logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(name)s: %(message)s'
)
logger = logging.getLogger("runtime-memory-agent")

# Initialize the agent core app
app = BedrockAgentCoreApp()

MODEL_ID = os.getenv('MODEL_ID')
MEMORY_ID = os.getenv('MEMORY_ID')
COGNITO_USER_POOL = os.getenv('COGNITO_USER_POOL')
IDENTITY_POOL_ID = os.getenv('IDENTITY_POOL_ID')
REGION = os.getenv('AWS_REGION')

# Global agent instance - will be initialized with first request
agent = None
memory_session_manager = None
actor_id = None

def get_aws_credentials_for_identity(identity_pool_id, id_token, region, user_pool_id):
    """
    Get temporary AWS credentials for a Cognito identity using a User Pool ID token
    """
    identity_client = boto3.client('cognito-identity', region_name=region)
    
    # Get ID from identity pool
    get_id_response = identity_client.get_id(
        IdentityPoolId=identity_pool_id,
        Logins={
            f'cognito-idp.{region}.amazonaws.com/{user_pool_id}': id_token
        }
    )
    identity_id = get_id_response['IdentityId']
    
    # Get credentials for the identity
    get_credentials_response = identity_client.get_credentials_for_identity(
        IdentityId=identity_id,
        Logins={
            f'cognito-idp.{region}.amazonaws.com/{user_pool_id}': id_token
        }
    )
    
    # Return the temporary credentials
    credentials = get_credentials_response['Credentials']
    return {
        'access_key_id': credentials['AccessKeyId'],
        'secret_key': credentials['SecretKey'],
        'session_token': credentials['SessionToken'],
        'expiration': credentials['Expiration'],
        'identity_id': identity_id
    }

class MemoryHookProvider(HookProvider):
    """Custom hook provider to integrate with Bedrock Memory"""
    
    def __init__(self):
        logger.info(f"Initializing MemoryHookProvider")
        # Use global memory_client that may have federated credentials
        self.memory_session_manager = memory_session_manager
    
    def on_agent_initialized(self, event: AgentInitializedEvent):
        """Load recent conversation history when agent starts"""
        logger.info("Agent initialization hook triggered")
        
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        logger.info(f"State values - actor_id: {actor_id}, session_id: {session_id}")
        
        if not all([actor_id, session_id]):
            logger.warning("Missing required state values")
            return
        
        try:
            # Check if the session exists
            logger.info(f"Checking if session {session_id} exists...")
            session_exists = False
            try:
                events = self.memory_session_manager.list_events(
                    actor_id=actor_id,
                    session_id=session_id,
                    max_results=1
                )
                session_exists = len(events) > 0
                logger.info(f"Session exists: {session_exists}")
            except Exception as e:
                logger.warning(f"Error checking session existence: {e}")
                session_exists = False
            
            if not session_exists:
                logger.info(f"No existing conversation found for session {session_id}")
                return
            
            # Load conversation history
            logger.info(f"Loading conversation history for existing session {session_id}")
            recent_turns = self.memory_session_manager.get_last_k_turns(
                actor_id=actor_id,
                session_id=session_id,
                k=5
            )            

            if recent_turns:
                logger.info(f"✅ Loaded {len(recent_turns)} conversation turns from memory")
                
                # Add messages to agent's conversation history
                for turn in reversed(recent_turns):
                    for message in turn:
                        role = message['role'].lower()  # 'user' or 'assistant'
                        parsed = ast.literal_eval(message['content']['text'])
                        content = parsed[0]['text']
                        
                        # Add to agent's message history
                        event.agent.messages.append({
                            "role": role,
                            "content": [{"text": content}]
                        })
                        logger.info(f"Added {role} message to history: {content[:50]}...")
                
                logger.info(f"✅ Added {len(event.agent.messages)} messages to conversation history")
            else:
                logger.info("No recent turns found for this session")
                    
        except Exception as e:
            logger.error(f"❌ Memory load error: {e}", exc_info=True)
    
    def on_message_added(self, event: MessageAddedEvent):
        """Store messages in memory"""
        logger.info("💬 Message added - storing in memory")
        
        actor_id = event.agent.state.get("actor_id")
        session_id = event.agent.state.get("session_id")
        
        if not all([actor_id, session_id]):
            logger.warning("Missing required state values")
            return
        
        try:
            messages = event.agent.messages
            last_message = messages[-1]
            message_content = str(last_message.get("content", ""))
            if last_message["role"] == "user":
                message_role = MessageRole.USER
            elif last_message["role"] == "assistant":
                message_role = MessageRole.ASSISTANT
            
            self.memory_session_manager.add_turns(
                actor_id=actor_id,
                session_id=session_id,
                messages=[ConversationalMessage(message_content, message_role)]
            )
            logger.info("✅ Message stored")
            
        except Exception as e:
            logger.error(f"❌ Error storing message: {e}")
    
    def register_hooks(self, registry: HookRegistry):
        logger.info("Registering memory hooks")
        registry.add_callback(MessageAddedEvent, self.on_message_added)
        registry.add_callback(AgentInitializedEvent, self.on_agent_initialized)

def initialize_agent(actor_id, session_id):
    """Initialize the agent for first use"""
    global agent
    
    logger.info(f"Initializing agent for actor_id={actor_id}, session_id={session_id}")
    
    # Create model and memory hook
    logger.info(f"Creating model with ID: {MODEL_ID}")
    model = BedrockModel(model_id=MODEL_ID)
    logger.info(f"Creating memory hook with region: {REGION}")
    memory_hook = MemoryHookProvider()
    
    # Create agent with proper initial state
    logger.info("Creating agent with memory hook")
    agent = Agent(
        model=model,
        hooks=[memory_hook],
        system_prompt="You're a helpful agent. You can remember previous interactions within the same session. Be friendly and concise in your responses.",
        state={
            "actor_id": actor_id,
            "session_id": session_id
        }
    )
    logger.info(f"✅ Agent initialized with state: {agent.state.get()}")

def log_cognito_sub_from_token(id_token):
    """Log the sub value from the Cognito ID token"""
    try:
        # Decode without verification (just for logging)
        decoded = jwt.decode(
            id_token,
            options={"verify_signature": False, "verify_aud": False, "verify_exp": False}
        )
        logger.info(f"ID Token sub claim: {decoded.get('sub')}")
        return decoded.get('sub')
    except Exception as e:
        logger.error(f"Error decoding token: {e}")
        return None

@app.entrypoint
def runtime_memory_agent(payload, context):
    """
    Main entry point for the memory-enabled agent with identity federation
    
    Args:
        payload: The input payload containing user data
        context: The runtime context object containing session information
    """
    global agent, memory_session_manager
    
    # Log both payload and context info
    logger.info(f"Received payload: {payload}")
    logger.info(f"Context: {context}")
    logger.info(f"Context Auth: {context.request_headers.get('Authorization')}")
    
    # Extract and validate required values
    user_input = payload.get("prompt")
    id_token = payload.get("id_token")  # Get the ID token from payload
    auth_header = context.request_headers.get('Authorization')
    session_id = context.session_id
    
    # Validate required fields
    if user_input is None:
        error_msg = "❌ ERROR: Missing 'prompt' field in payload"
        logger.error(error_msg)
        return error_msg
    
    # Set up federated identity if ID token is provided
    if id_token and IDENTITY_POOL_ID:
        logger.info("ID token provided - setting up federated identity")
        
        # Get AWS credentials using the ID token
        user_credentials = get_aws_credentials_for_identity(
            identity_pool_id=IDENTITY_POOL_ID,
            id_token=id_token,
            region=REGION,
            user_pool_id=COGNITO_USER_POOL
        )
        
        # Set up actor_id
        logger.info(f"Identity Credentials: {user_credentials['identity_id']}")
        actor_id = user_credentials['identity_id']

        # Set up boto3 session with federated credentials
        session = boto3.Session(
            aws_access_key_id=user_credentials['access_key_id'],
            aws_secret_access_key=user_credentials['secret_key'],
            aws_session_token=user_credentials['session_token'],
            region_name=REGION
        )
        
        log_cognito_sub_from_token(id_token)

        # Create memory client with federated credentials 
        memory_session_manager = MemorySessionManager(
            memory_id = MEMORY_ID,
            region_name = REGION, 
            boto3_session = session
        )
        logger.info("✅ Successfully configured federated credentials for memory operations")
            
    
    # Initialize agent on first request
    if agent is None:
        logger.info("First request - initializing agent")
        initialize_agent(actor_id, session_id)
    else:
        logger.info("Using existing agent instance")
        # Update the session ID in case it changed
        if agent.state.get("session_id") != session_id:
            logger.info(f"Updating session ID to {session_id}")
            agent.state.set("session_id", session_id)
        if agent.state.get("actor_id") != actor_id:
            logger.info(f"Updating actor ID to {actor_id}")
            agent.state.set("actor_id", actor_id)
    
    # Invoke the agent with the user's input
    logger.info(f"Invoking agent with input: {user_input}")
    response = agent(user_input)
    response_text = response.message['content'][0]['text']
    logger.info(f"✅ Agent response: {response_text[:50]}…")
    
    return response_text

if __name__ == "__main__":
    logger.info("Starting AgentCore application")
    app.run()

## 4. Implantando no AgentCore Runtime

Nesta seção, implantaremos nosso agente no Amazon Bedrock AgentCore Runtime, um ambiente de runtime gerenciado para agentes que oferece escalabilidade e operações simplificadas. O AgentCore Runtime lida com a complexidade da infraestrutura, permitindo que você se concentre na lógica do seu agente em vez de preocupações com a implantação.

Diferente dos métodos tradicionais de implantação que requerem configuração e gerenciamento manual de servidores, o AgentCore Runtime implanta seus containers de agente na infraestrutura AWS e fornece endpoints HTTPS seguros para invocação. Essa abordagem garante que seu agente possa escalar com a demanda e operar de forma confiável em ambientes de produção.

> 💡 **Dica**: O AgentCore starter toolkit cuida de todas as etapas complexas de implantação para nós, incluindo IAM roles, repositórios ECR e builds de containers.

### Configurar a Implantação

Vamos configurar nossa implantação:

In [ ]:
iam_role = create_agentcore_role(agent_name=f"runtime_memory_agent_{unique_id}", region=REGION)

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
import time

agentcore_runtime = Runtime()
agent_name = f"runtime_memory_agent_{unique_id}"

response = agentcore_runtime.configure(
    entrypoint="runtime_identity_memory_agent.py", 
    execution_role = iam_role["Role"]["RoleName"],
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=REGION,
    agent_name=agent_name,
    non_interactive=True, 
    memory_mode="NO_MEMORY",
    idle_timeout = 60,
    request_header_configuration = {"requestHeaderAllowlist": ["Authorization"]},
    authorizer_configuration={
        "customJWTAuthorizer": {
            "discoveryUrl": cognito_config.get("discovery_url"),
            "allowedClients": [cognito_config.get("client_id")]
        }
    }
)
response

### Iniciar o agente

Agora vamos iniciar nosso agente no AgentCore Runtime. Esta etapa pega nosso agente configurado e o implanta na infraestrutura gerenciada do AgentCore Runtime. Durante este processo, também estamos passando as variáveis de ambiente essenciais que nosso agente precisa: o memory ID que criamos anteriormente, o model ID a ser usado, a região AWS e o Cognito user pool ID para autenticação.

Uma vez implantado, nosso agente será acessível através de um endpoint seguro que podemos invocar com mensagens de usuário. O endpoint será protegido pela autenticação de entrada do AgentCore Identity, garantindo que apenas usuários autorizados possam acessar nosso agente.

In [ ]:
launch_result = agentcore_runtime.launch(
    env_vars={
        "MEMORY_ID": MEMORY_ID,
        "MODEL_ID": "us.anthropic.claude-haiku-4-5-20251001-v1:0",
        "AWS_REGION": REGION,
        "COGNITO_USER_POOL": cognito_config["pool_id"],
        "IDENTITY_POOL_ID":cognito_config["identity_pool_id"]
    }
)

### Verificar status da implantação

Vamos verificar o status de implantação do nosso agente. Isso pode levar alguns minutos enquanto o AgentCore Runtime constrói seu container, provisiona os recursos necessários e implanta seu agente na infraestrutura AWS. Faremos polling do status a cada 10 segundos até que a implantação seja concluída.

In [ ]:
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']

while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(f"Current status: {status}")

if status == 'READY':
    print("✅ Agent successfully deployed!")
else:
    print(f"❌ Deployment ended with status: {status}")

account_id = status_response.config.account
role_arn = status_response.config.execution_role
role_name = role_arn.split("/")[-1]
print(f"Account ID is {account_id}")
print(f"Agent role is {role_arn}")
print(f"Role name is {role_name}")

## 5. Testando Seu Agente

Agora que nosso agente está implantado, vamos testá-lo enviando mensagens e verificando se ele consegue lembrar interações anteriores. Também testaremos que diferentes usuários têm contextos de memória isolados, garantindo que a conversa de um usuário não seja visível para outro usuário.

**Notas Importantes sobre Gerenciamento de Sessão**

- **Gerenciamento de Sessão**: Embora o AgentCore Runtime gere automaticamente um session ID se nenhum for fornecido, é recomendado gerenciar explicitamente os session IDs na sua aplicação. Isso lhe dá melhor controle sobre:
  - Continuar conversas após timeouts de sessão
  - Criar novas sessões quando apropriado (ex.: usuário inicia uma nova conversa)
  - Lidar com múltiplas conversas paralelas com o mesmo usuário
  - Implementar políticas de expiração de sessão com base nas necessidades da sua aplicação

- **Persistência de Memória**: Mesmo se uma sessão expirar no AgentCore Runtime, nosso agente pode recuperar conversas anteriores do AgentCore Memory quando uma nova sessão é iniciada com o mesmo usuário.

In [ ]:
import time

def test_user_memory_isolation_with_federated_identity():
    """
    Test user memory isolation using federated identity credentials.
    """
    print("\n" + "=" * 50)
    print("USER MEMORY ISOLATION WITH FEDERATED IDENTITY TEST")
    print("=" * 50)
    
    # Extract bearer tokens and ID tokens
    testuser1_token = cognito_config["bearer_tokens"]["testuser1"]
    testuser2_token = cognito_config["bearer_tokens"]["testuser2"]
    testuser1_id_token = cognito_config["id_tokens"]["testuser1"]
    testuser2_id_token = cognito_config["id_tokens"]["testuser2"]
    
    # Create unique session IDs for each test phase
    user1_session_id = f"memory-agent-session-user1-{int(time.time())}"
    user2_session_id = f"memory-agent-session-user2-{int(time.time())}"
    
    # PHASE 1: Test with user1's memory persistence
    print("\n" + "=" * 50)
    print("PHASE 1: USER 1 MEMORY PERSISTENCE TEST")
    print("=" * 50)

    # Step 1: User 1 shares initial information
    print("\n" + "-" * 50)
    print("STEP 1: User 1 shares information")
    print("-" * 50)

    user1_prompt1 = "My name is Dani and my favorite color is blue."
    response1 = agentcore_runtime.invoke(
        {
            "prompt": user1_prompt1,
            "id_token": testuser1_id_token
        },
        session_id=user1_session_id,
        bearer_token=testuser1_token
    )
    print(f"User 1 prompt: \"{user1_prompt1}\"")
    print(f"User 1 response: \"{response1['response']}\"")

    # Wait for session to terminate (75 seconds)
    print("\nWaiting 75 seconds for session to terminate...")
    time.sleep(75)

    # Step 2: User 1 asks to recall information
    print("\n" + "-" * 50)
    print("STEP 2: User 1 recalls information (should succeed)")
    print("-" * 50)

    user1_prompt2 = "What is my name and favorite color?"
    response2 = agentcore_runtime.invoke(
        {
            "prompt": user1_prompt2,
            "id_token": testuser1_id_token
        },
        session_id=user1_session_id,
        bearer_token=testuser1_token
    )
    print(f"User 1 prompt: \"{user1_prompt2}\"")
    print(f"User 1 response: \"{response2['response']}\"")

    # PHASE 2: Test user2 memory isolation
    print("\n" + "=" * 50)
    print("PHASE 2: USER 2 MEMORY ISOLATION TEST")
    print("=" * 50)

    # Step 3: User 2 shares information
    print("\n" + "-" * 50)
    print("STEP 3: User 2 shares information")
    print("-" * 50)

    user2_prompt1 = "My name is Paula and my favorite color is pink."
    response3 = agentcore_runtime.invoke(
        {
            "prompt": user2_prompt1,
            "id_token": testuser2_id_token
        },
        session_id=user2_session_id,
        bearer_token=testuser2_token
    )
    print(f"User 2 prompt: \"{user2_prompt1}\"")
    print(f"User 2 response: \"{response3['response']}\"")

    # Wait for session to terminate
    print("\nWaiting 75 seconds for session to terminate...")
    time.sleep(75)

    # Step 4: User 2 tries to recall (should only see their own info)
    print("\n" + "-" * 50)
    print("STEP 4: User 2 recalls information (should see only their data)")
    print("-" * 50)

    user2_prompt2 = "What is my name and favorite color?"
    response4 = agentcore_runtime.invoke(
        {
            "prompt": user2_prompt2,
            "id_token": testuser2_id_token
        },
        session_id=user2_session_id,
        bearer_token=testuser2_token
    )
    print(f"User 2 prompt: \"{user2_prompt2}\"")
    print(f"User 2 response: \"{response4['response']}\"")
    print("\n✅ Each user should only see their own information, demonstrating memory isolation")

In [ ]:
test_user_memory_isolation_with_federated_identity()

## Conceitos Principais

Neste tutorial, você aprendeu vários conceitos importantes para construir agentes habilitados com memória no AgentCore:

1. **Integração de Memória**: Como usar o AgentCore Memory para armazenar o histórico de conversas entre sessões, permitindo que seu agente mantenha o contexto ao longo do tempo mesmo quando as sessões expiram.

2. **Gerenciamento de Sessão**: Como usar session IDs para organizar conversas e recuperar o histórico relevante quando um usuário retorna, criando uma experiência contínua.

3. **Implantação no AgentCore Runtime**: Como implantar seu agente em um ambiente de runtime de produção que lida com escalabilidade, segurança e gerenciamento de infraestrutura automaticamente.

4. **Memory Hooks**: Como implementar hooks personalizados que se integram com serviços de memória, permitindo armazenar e recuperar o histórico de conversas em pontos específicos do ciclo de vida do agente.

5. **Federated Identity e Privacidade**: Como usar Cognito Identity Pools e credenciais federadas para garantir que o histórico de conversas de cada usuário seja privado e isolado de outros usuários através do AWS IAM.

Esses conceitos fornecem uma base para construir agentes mais complexos com memória persistente e capacidades sofisticadas de gerenciamento de conversas.

## Limpeza (Opcional)

Se você não precisar mais dos recursos criados neste tutorial, pode limpá-los para evitar cobranças desnecessárias da AWS.

In [ ]:
# Only run this cell if you want to delete all resources

# 1. Delete the AgentCore Runtime
if 'launch_result' in locals() and hasattr(launch_result, 'agent_id'):
    try:
        agentcore_control_client = boto3.client(
            'bedrock-agentcore-control',
            region_name=REGION
        )
        
        runtime_delete_response = agentcore_control_client.delete_agent_runtime(
            agentRuntimeId=launch_result.agent_id,
        )
        print(f"✅ Deleted AgentCore Runtime: {launch_result.agent_id}")
    except Exception as e:
        print(f"❌ Error deleting AgentCore Runtime: {e}")
else:
    print("No AgentCore Runtime to delete")

# 2. Delete the ECR repository
if 'launch_result' in locals() and hasattr(launch_result, 'ecr_uri'):
    try:
        ecr_client = boto3.client(
            'ecr',
            region_name=REGION
        )
        
        repository_name = launch_result.ecr_uri.split('/')[1]
        response = ecr_client.delete_repository(
            repositoryName=repository_name,
            force=True  # Force deletion even if it contains images
        )
        print(f"✅ Deleted ECR repository: {repository_name}")
    except Exception as e:
        print(f"❌ Error deleting ECR repository: {e}")
else:
    print("No ECR repository to delete")

# 3. Delete the memory resource
try:
    memory_manager.delete_memory(memory_id=MEMORY_ID)
    print(f"✅ Deleted memory resource: {MEMORY_ID}")
except Exception as e:
    print(f"❌ Error deleting memory resource: {e}")

# 4. Delete the Cognito User Pool and associated resources
if 'cognito_config' in locals() and cognito_config and 'pool_id' in cognito_config:
    try:
        cognito_client = boto3.client('cognito-idp', region_name=REGION)
        
        # Get the user pool ID
        pool_id = cognito_config['pool_id']
        
        # List and delete all user pool clients
        clients_response = cognito_client.list_user_pool_clients(
            UserPoolId=pool_id,
            MaxResults=60
        )
        
        for client in clients_response.get('UserPoolClients', []):
            client_id = client['ClientId']
            cognito_client.delete_user_pool_client(
                UserPoolId=pool_id,
                ClientId=client_id
            )
            print(f"✅ Deleted User Pool Client: {client_id}")
        
        # Delete the user pool itself
        cognito_client.delete_user_pool(
            UserPoolId=pool_id
        )
        print(f"✅ Deleted Cognito User Pool: {pool_id}")
        
    except Exception as e:
        print(f"❌ Error deleting Cognito resources: {e}")
else:
    print("No Cognito resources to delete")

# 5. Function to delete IAM role and all its versions
def delete_iam_role(role_identifier, region=REGION):
    """
    Deletes an IAM role including all attached policies and versions
    
    Args:
        role_identifier (str): The ARN or name of the IAM role
        region (str): AWS region
    """
    try:
        iam_client = boto3.client('iam', region_name=region)
        
        # Determine if the identifier is an ARN or a role name
        if role_identifier.startswith('arn:aws:iam::'):
            # Extract role name from ARN
            role_name = role_identifier.split('/')[-1]
        else:
            role_name = role_identifier
            
        print(f"Attempting to delete IAM role: {role_name}")
            
        # 1. Detach all managed policies
        attached_policies = iam_client.list_attached_role_policies(RoleName=role_name)
        for policy in attached_policies.get('AttachedPolicies', []):
            iam_client.detach_role_policy(
                RoleName=role_name,
                PolicyArn=policy['PolicyArn']
            )
            print(f"✅ Detached managed policy: {policy['PolicyArn']}")
            
        # 2. Delete all inline policies
        inline_policies = iam_client.list_role_policies(RoleName=role_name)
        for policy_name in inline_policies.get('PolicyNames', []):
            iam_client.delete_role_policy(
                RoleName=role_name,
                PolicyName=policy_name
            )
            print(f"✅ Deleted inline policy: {policy_name}")
            
        # 3. Delete any instance profiles associated with the role
        instance_profiles = iam_client.list_instance_profiles_for_role(RoleName=role_name)
        for profile in instance_profiles.get('InstanceProfiles', []):
            iam_client.remove_role_from_instance_profile(
                InstanceProfileName=profile['InstanceProfileName'],
                RoleName=role_name
            )
            print(f"✅ Removed role from instance profile: {profile['InstanceProfileName']}")
            
        # 4. Finally delete the role
        iam_client.delete_role(RoleName=role_name)
        print(f"✅ Successfully deleted IAM role: {role_name}")
        
    except Exception as e:
        print(f"❌ Error deleting IAM role: {e}")


delete_iam_role(role_arn)

print("\n✅ Cleanup complete")